# Ch 6. 그래프 만들고 멀티홉 질의하기 — hands-on

지식 그래프와 GraphRAG 스터디 · [study-graphrag](https://desty.github.io/study-graphrag/)

이 노트북은 **설치 한 줄로 Colab에서 바로 돈다.** 그래프 DB 서버 없이 `networkx`로 [Ch 6](https://desty.github.io/study-graphrag/part3/06-neo4j-cypher/) 본문과 똑같은 그래프를 만들고, 벡터 RAG가 못 풀던 **2홉 질의**를 실제로 실행한다. 프로덕션용 Neo4j/Cypher 코드는 맨 아래에 함께 둔다.

In [ ]:
!pip -q install networkx
import networkx as nx

# 속성 그래프(LPG)처럼: 노드/엣지에 속성을 붙인다 (MultiDiGraph = 방향 + 다중 엣지)
G = nx.MultiDiGraph()
G.add_node("이수진", label="Customer", tier="gold")
G.add_node("order101", label="Order", total=52000)
G.add_node("키보드", label="Product")
G.add_node("마우스", label="Product")
G.add_edge("이수진", "order101", rel="PLACES")
G.add_edge("order101", "키보드", rel="CONTAINS", qty=1, price=40000)
G.add_edge("order101", "마우스", rel="CONTAINS", qty=1, price=12000)
print(f"노드 {G.number_of_nodes()}개, 엣지 {G.number_of_edges()}개")

## 멀티홉 질의 — "이수진이 산 상품은?" (Customer→Order→Product, 2홉)

In [ ]:
def neighbors_by_rel(g, node, rel):
    """node에서 rel 관계로 나가는 이웃들"""
    return [v for _, v, d in g.out_edges(node, data=True) if d.get("rel") == rel]

def products_bought(g, customer):
    products = []
    for order in neighbors_by_rel(g, customer, "PLACES"):      # 1홉: 고객→주문
        products += neighbors_by_rel(g, order, "CONTAINS")      # 2홉: 주문→상품
    return products

print(products_bought(G, "이수진"))   # ['키보드', '마우스']  ← 관계를 따라가면 끝

## 3홉 추천 — "같은 상품을 산 다른 고객" (지금은 데이터가 작아 비어 있다; 노드를 더 넣어 보라)

In [ ]:
def co_buyers(g, customer):
    mine = set(products_bought(g, customer))
    others = set()
    for c in [n for n, d in g.nodes(data=True) if d.get("label") == "Customer" and n != customer]:
        if mine & set(products_bought(g, c)):
            others.add(c)
    return others

# 연습: 고객 한 명과 주문/상품을 더 add_edge 해서 추천이 채워지는지 보라
print(co_buyers(G, "이수진"))

## 프로덕션 — 같은 그래프를 Neo4j/Cypher로

위 networkx 데모는 개념을 보이려는 것이고, 실전은 그래프 DB를 쓴다. Neo4j(도커 또는 Aura 무료)에 연결하면 같은 그래프가 이렇게 된다:

```python
# pip install neo4j ; docker run -p7687:7687 -e NEO4J_AUTH=neo4j/testpassword neo4j:5
from neo4j import GraphDatabase
driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "testpassword"))
def run(cypher, **p):
    with driver.session() as s: return list(s.run(cypher, **p))

run('''MERGE (c:Customer {name:'이수진'})-[:PLACES]->(o:Order {id:101})
       MERGE (o)-[:CONTAINS]->(:Product {name:'키보드'})''')

# 같은 2홉 질의 — Cypher 한 줄
run('''MATCH (c:Customer {name:'이수진'})-[:PLACES]->(:Order)-[:CONTAINS]->(p:Product)
       RETURN p.name''')
```

networkx의 `out_edges` 순회가 Cypher의 `-[:PLACES]->(:Order)-[:CONTAINS]->` 패턴과 같은 일을 한다.